# Anvendt KI og ML: Klassifisering

Velkommen til denne notebooken som inngår i Høyskolen Kristianias emne «Anvendt kunstig intelligens og maskinlæring», modul 2, «Veiledet læring».

## Hvordan du bruker denne notebooken
Dette er en Jupyter notebook med tekst og Python-kode som du kjører ved å trykke shift-enter. Hensikten er å bli kjent med relevante konsepter i praksis, ikke nødvendigvis å forstå alle detaljer i koden. Ta deg god tid til å lese teksten, se gjennom koden, kjøre kodecellene (pass på riktig rekkefølge), studere resultatene, og tenke gjennom hva vi gjør, hvorfor og hvordan.

Underveis i teksten vil du finne referanser til Canvas-oppgaver _i kursiv_. Gå til Canvas og besvar spørsmålene der før du går videre i notebooken.

## Problemstilling: Klassifisering av brystkreft-celleprøver
Vi skal trene en modell til å skjelne mellom ondartede og godartede brystkreft-celleprøver. Dette er et klassisk multi-label klassifiseringsproblem.


In [ ]:
#Import relevant packages

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn import datasets  # scikit-learn dataset library
from sklearn.model_selection import train_test_split  # train_test_split function
from sklearn import svm  # svm model
from sklearn import metrics  # scikit-learn metrics module for accuracy calculation

In [ ]:
def make_histogram(data, variable, binwidth=None, lower_lim=None, upper_lim=None):
    if upper_lim == None:
        upper_lim = data[variable].max()
    if lower_lim == None:
        lower_lim=data[variable].min()
    if binwidth != None:
        bins = np.arange(lower_lim, upper_lim+binwidth, binwidth)
    else:
        bins = None
    plt.hist(data[variable], bins=bins, edgecolor='black', alpha=0.75, range=[data[variable].min(), upper_lim])
    plt.ylabel("antall")
    plt.xlabel(variable)
    plt.show()

## Data
Til å bygge modellen bruker vi et standard-datasett fra scikit-learn, nemlig brystkreft-datasettet. Dette datasettet beskriver ulike karakteristika ved cellekjerner i brystvev.

Datasettet har 30 features: mean radius, mean texture, mean perimeter, mean area, mean smoothness, mean compactness, mean concavity, mean concave points, mean symmetry, mean fractal dimension, radius error, texture error, perimeter error, area error, smoothness error, compactness error, concavity error, concave points error, symmetry error, fractal dimension error, worst radius, worst texture, worst perimeter, worst area, worst smoothness, worst compactness, worst concavity, worst concave points, worst symmetry, worst fractal dimension og type of cancer.

Sistnevnte, dvs. type of cancer, angir hvilken type kreft prøven stammer fra. Det er to klasser, ondartet (harmful) og godartet (not harmful), og modellen vår skal trenes til å skjelne mellom disse to typene.

Vi følger eksempelet og forklaringen fra https://www.datacamp.com/community/tutorials/svm-classification-scikit-learn-python. Se gjerne lenken for flere detaljer.

Vi skal benytte metoden støttevektormaskiner (support vector machines, SVM) som du kan lese mer om på https://scikit-learn.org/stable/modules/svm.html.

### Laste inn og utforske dataene

In [ ]:
#Load dataset
cancer = datasets.load_breast_cancer()

#Data is loaded as a dictionary but is easier to work with as a pandas data frame. We define a function to convert between the two
def sklearn_to_df(sklearn_dataset):
    df = pd.DataFrame(sklearn_dataset.data, columns=sklearn_dataset.feature_names)
    df['target'] = pd.Series(sklearn_dataset.target)
    return df

#And apply it
df = sklearn_to_df(cancer)

In [ ]:
df

Disse dataene er allerede pre-prosseserte. Ved å plotte histogrammer av de ulike variablene, vil vi se at outliers, urealistiske verdier osv. er fjernet. Merk at ondartet har annotasjonen 0 og godartet 1.

In [ ]:
#Plot data to check for outliers
all_col = df.select_dtypes(include=np.number).columns.tolist()  # selecting numeric columns
for col in all_col:
    make_histogram(df, col)


### Splitte dataene
For å kunne kvantifisere modellens ytelse, splitter vi datasettet i et treningssett og et testsett. Valideringsdata er vanligvis et subsett av treningsdataene. I denne notebooken kommer vi ikke til å gjøre hyperparametertuning, så vi definerer ikke valideringsdata eksplisitt. Vi benytter scikit-learns funksjon for å splitte data. Vi må spesifisere input features (X), output features (y, target) og test_size, dvs. hvilken andel av dataene som skal brukes til test.

In [ ]:
#Split dataset into training set and test set
X_train, X_test, y_train, y_test = train_test_split(df.drop('target', axis=1), df['target'], test_size=0.3, random_state=109) 
# test_size=0.3 corresponds to 70% training and 30% test

## Sette opp modellen, trene og predikere
For å trene modellen må vi først importere SVM-modulen og initialisere SVM-klassifiseringsobjektet med et filter (kernel) som vi tror vil passe til problemstillingen. Filteret sier noe om hvor fleksibel (wiggly) inndelingen mellom klasser kan være. Med et lineært filter kan inndelingen gjøres med rette linjer. Med andre filtre kan man oppnå buede eller sikksakkete inndelinger. Vi velger et lineært filter. Deretter tilpasser vi modellen til treningsdataene, og så predikerer vi på testdata.

_Oppgave: Svar på flervalgsspørsmål 1 (vist nedenfor) i Canvas._

1. _Hvordan fungerer en støttevektormaskin (support vector machine, SVM)?_

In [ ]:
#Create a svm Classifier
clf = svm.SVC(kernel='linear') # Linear Kernel

#Train the model using the training sets
clf.fit(X_train, y_train)

#Predict the response for the test dataset
y_pred = clf.predict(X_test)

In [ ]:
x = range(len(y_test))
index = np.argsort(y_test)
plt.scatter(x, y_test.iloc[index], alpha=1, marker='x', s=70, label='Test values')
plt.scatter(x, y_pred[index], alpha=0.5, label='Predicted values')
plt.legend()

Her ser vi at modellen predikerer rett klasse i de fleste tilfellene, siden de oransje sirklene ligger oppå de blå kryssene. Men det er noen tilfeller hvor modellen predikerer feil.

## Evaluering av modellen
### Nøyaktighet (accuracy)
Til å evaluere modellen kan vi benytte _nøyaktighet_ som kvantifiserer andelen av korrekt klassifiserte elementer som «antall sanne positive og sanne negative prediksjoner, dividert med antall prediksjoner».

Sann positiv = modellen predikerer at en gitt celleprøve er godartet når den faktisk er det. \
Sann negativ = modellen predikerer at en gitt celleprøve er ondartet når den faktisk er det. \
Falsk positiv = modellen predikerer at en gitt celleprøve er godartet når den faktisk er ondartet. \
Falsk negativ = modellen predikerer at en gitt celleprøve er ondartet når den faktisk er godartet. 

Det er dog et problem at definisjonen av nøyaktighet innebærer at falske positive og falske negative har samme kvalitative verdi.

La oss anta at vi hadde trent på et datasett med 100&nbsp;000 celleprøver hvorav bare 0,1&nbsp;% var ondartet. Mest sannsynlig ville modellen bare lært seg å predikere «godartet» for alle celleprøvene. Nøyaktigheten ville da bli 99,9&nbsp;%. Men den ville ikke plukke opp et eneste tilfelle med ondartet kreft. Hvis modellen i stedet hadde predikert de 0,1&nbsp;% tilfellene med ondartet kreft korrekt og 0,1 % falske positiver, ville den fortsatt ha en nøyaktighet på 99,9&nbsp;%. Sistenevnte modell er mye nyttigere enn den første modellen, men vi kan ikke skjelne mellom de to basert på nøyaktighet. Derfor trenger vi bedre evalueringskriterier.


### Presisjon og sensitivitet (precision and recall)
I stedet benytter vi _presisjon_ og _sensitivitet_. Noen steder brukes ordet «tilbakekall» om sensitivitet. Presisjon sier hvor stor andel av prediksjonene som er klassifisert som godartet (1), som faktisk er korrekt klassifisert. Og sensitivitet sier hvor stor andel av de som egentlig er godartet (1), som modellen klassifiserte korrekt.

En god modell er da karakteriseret ved både høy presisjon og høy sensitivitet.

Se gjerne den fine illustrasjon fra Wikipedia på forskjellen mellom nøyaktighet, presisjon og sensitivitet: https://en.wikipedia.org/wiki/Precision_and_recall.

In [ ]:
# Model Accuracy: how often is the classifier correct?
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
# Model Precision: what percentage of positive tuples are labeled as such?
print("Precision:",metrics.precision_score(y_test, y_pred))
# Model Recall: what percentage of positive tuples are labelled as such?
print("Recall:",metrics.recall_score(y_test, y_pred))

_Oppgave: Svar på fritekstspørsmål 2 (vist nedenfor) i Canvas._

2. _Er dette en bra modell?_